# Diurnal Population Simulation (Gachibowli) — UI + Monte Carlo

This notebook provides an **ipywidgets UI** to:
- Load building + boundary shapefiles
- Run a single diurnal simulation
- Run **Monte Carlo** (uncertainty over work times + workplace assignment)
- Visualize time series + maps + uncertainty bands
- Export outputs to `outputs/`


In [ ]:
import sys
from pathlib import Path

# Ensure repo root is on sys.path so we can import src.diurnal_sim
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

import ipywidgets as widgets
from IPython.display import display, clear_output

from src.diurnal_sim.io import load_buildings_and_boundary, validate_buildings_schema
from src.diurnal_sim.engine import DiurnalModel, DiurnalModelConfig
from src.diurnal_sim.mc import MonteCarloConfig, run_monte_carlo
from src.diurnal_sim.viz import plot_total_timeseries, plot_uncertainty_band, plot_map_snapshot
from src.diurnal_sim.export import export_simulation
from src.diurnal_sim.animation import save_map_animation

print('Ready.')


## 1) UI Controls

In [ ]:
# Default paths (match the prototype notebook)
default_buildings = str((ROOT / 'Buildings' / 'WEST_ZONE_BLDG_HGT_POP_MEAN.shp').resolve())
default_boundary = str((ROOT / 'Boundary' / 'UTM44N_GCKMMHS_ZONE_POLY_DSLV_EXTN.shp').resolve())
default_outputs = str((ROOT / 'outputs').resolve())

buildings_path = widgets.Text(value=default_buildings, description='Buildings:', layout=widgets.Layout(width='95%'))
boundary_path = widgets.Text(value=default_boundary, description='Boundary:', layout=widgets.Layout(width='95%'))
outputs_dir = widgets.Text(value=default_outputs, description='Outputs:', layout=widgets.Layout(width='95%'))

day_of_week = widgets.Dropdown(options=['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'], value='Monday', description='Day:')
time_interval = widgets.Dropdown(options=[0.25, 0.5, 1.0], value=0.5, description='Δt (h):')
worker_share = widgets.FloatSlider(value=0.40, min=0.05, max=0.90, step=0.05, description='Worker share:', readout_format='.2f', continuous_update=False)
distance_decay = widgets.IntSlider(value=2000, min=200, max=8000, step=200, description='Decay (m):', continuous_update=False)
seed = widgets.IntText(value=0, description='Seed:')

n_runs = widgets.IntSlider(value=30, min=5, max=200, step=5, description='MC runs:', continuous_update=False)

btn_load = widgets.Button(description='Load data', button_style='primary')
btn_single = widgets.Button(description='Run single', button_style='success')
btn_mc = widgets.Button(description='Run Monte Carlo', button_style='warning')
btn_export = widgets.Button(description='Export single', button_style='')
btn_anim = widgets.Button(description='Save MP4 (single)', button_style='')

out_status = widgets.Output(layout={'border': '1px solid #ddd'})
out_plots = widgets.Output()

controls = widgets.VBox([
    buildings_path,
    boundary_path,
    outputs_dir,
    widgets.HBox([day_of_week, time_interval, seed]),
    worker_share,
    distance_decay,
    n_runs,
    widgets.HBox([btn_load, btn_single, btn_mc, btn_export, btn_anim]),
    out_status,
])

display(controls, out_plots)


## 2) Backend callbacks

In [ ]:
STATE = {
    'buildings': None,
    'boundary': None,
    'single_result': None,
    'mc_summary': None,
}

def _ensure_projected(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    # If CRS is missing or geographic, reproject to UTM 44N (Hyderabad) for metric distance calculations.
    if gdf.crs is None:
        raise ValueError('Buildings CRS is missing. Please assign a CRS before running.')
    try:
        if gdf.crs.is_geographic:
            return gdf.to_crs('EPSG:32644')
    except Exception:
        # If CRS object doesn't provide is_geographic (older pyproj), skip.
        pass
    return gdf

def on_load(_):
    with out_status:
        clear_output()
        try:
            data = load_buildings_and_boundary(
                buildings_path=buildings_path.value,
                boundary_path=boundary_path.value or None,
            )
            validate_buildings_schema(data.buildings, lulc_col='LU_B_PRJ', pop_col='B_POP_SHAR')

            buildings = _ensure_projected(data.buildings)
            boundary = data.boundary
            if boundary is not None and boundary.crs is not None and boundary.crs != buildings.crs:
                boundary = boundary.to_crs(buildings.crs)

            STATE['buildings'] = buildings
            STATE['boundary'] = boundary
            STATE['single_result'] = None
            STATE['mc_summary'] = None

            print(f"✓ Buildings loaded: {len(buildings):,} | CRS: {buildings.crs}")
            if boundary is not None:
                print('✓ Boundary loaded')
            else:
                print('• Boundary not loaded (optional)')

            # Quick overview
            print('Land-use counts:')
            print(buildings['LU_B_PRJ'].value_counts().head(10))
        except Exception as e:
            print('✗ Load failed:', e)

def _make_model_config() -> DiurnalModelConfig:
    return DiurnalModelConfig(
        day_of_week=day_of_week.value,
        time_interval_hours=float(time_interval.value),
        worker_share=float(worker_share.value),
        distance_decay_m=float(distance_decay.value),
    )

def on_run_single(_):
    with out_status:
        clear_output()
        if STATE['buildings'] is None:
            print('Load data first.')
            return
        cfg = _make_model_config()
        model = DiurnalModel(STATE['buildings'], config=cfg)
        res = model.run(seed=int(seed.value))
        STATE['single_result'] = res
        STATE['mc_summary'] = None
        print('✓ Single simulation complete.')

    with out_plots:
        clear_output()
        fig, ax = plt.subplots(1, 1, figsize=(12, 4))
        plot_total_timeseries(res.timesteps, res.total_population_series(), ax=ax)
        plt.show()

        # Default map snapshot at 12:00
        hour = 12.0
        idx = int(round(hour / float(cfg.time_interval_hours)))
        idx = max(0, min(idx, res.population_matrix.shape[0]-1))
        fig, ax = plt.subplots(1, 1, figsize=(10, 8))
        plot_map_snapshot(res.buildings, res.population_matrix[idx, :], ax=ax, title=f'Snapshot @ {hour:0.0f}h')
        if STATE['boundary'] is not None:
            STATE['boundary'].plot(ax=ax, facecolor='none', edgecolor='cyan', linewidth=2)
        plt.show()

def on_run_mc(_):
    with out_status:
        clear_output()
        if STATE['buildings'] is None:
            print('Load data first.')
            return
        cfg = _make_model_config()
        mc_cfg = MonteCarloConfig(n_runs=int(n_runs.value), base_seed=int(seed.value))
        summ = run_monte_carlo(buildings=STATE['buildings'], model_config=cfg, mc_config=mc_cfg)
        STATE['mc_summary'] = summ
        STATE['single_result'] = None
        print('✓ Monte Carlo complete.')
        print(summ.meta)

    with out_plots:
        clear_output()
        p5, p50, p95 = summ.total_population_percentiles
        fig, ax = plt.subplots(1, 1, figsize=(12, 4))
        plot_uncertainty_band(summ.timesteps, p5, p50, p95, ax=ax)
        plt.show()

        # Map: median at 12:00 (if stored)
        if summ.building_percentiles_by_hour is not None and 12.0 in summ.key_hours:
            hour_i = list(summ.key_hours).index(12.0)
            p5_b, p50_b, p95_b = summ.building_percentiles_by_hour[hour_i]
            fig, ax = plt.subplots(1, 1, figsize=(10, 8))
            plot_map_snapshot(STATE['buildings'], p50_b, ax=ax, title='MC median snapshot @ 12h')
            if STATE['boundary'] is not None:
                STATE['boundary'].plot(ax=ax, facecolor='none', edgecolor='cyan', linewidth=2)
            plt.show()

def on_export(_):
    with out_status:
        clear_output()
        res = STATE.get('single_result')
        if res is None:
            print('Run a single simulation first (Export currently exports the single run).')
            return
        out = Path(outputs_dir.value) / 'single'
        paths = export_simulation(result=res, output_dir=out)
        print('✓ Exported:')
        print('  ', paths.timeseries_csv)
        print('  ', paths.matrix_npy)
        print('  ', paths.metadata_json)
        print('  ', paths.buildings_gpkg if paths.buildings_gpkg else '(GeoPackage export skipped)')

def on_anim(_):
    with out_status:
        clear_output()
        res = STATE.get('single_result')
        if res is None:
            print('Run a single simulation first.')
            return
        out = Path(outputs_dir.value) / 'single' / 'population.mp4'
        save_map_animation(result=res, mp4_path=out)
        print('✓ Saved MP4:', out)

btn_load.on_click(on_load)
btn_single.on_click(on_run_single)
btn_mc.on_click(on_run_mc)
btn_export.on_click(on_export)
btn_anim.on_click(on_anim)


## 3) Interactive map viewer (after a run)

After you run **Single** or **Monte Carlo**, use these widgets to browse map snapshots.

In [ ]:
hour_view = widgets.FloatSlider(value=12.0, min=0.0, max=24.0, step=0.5, description='Hour:', continuous_update=False)
mc_stat = widgets.Dropdown(options=['p5','p50','p95'], value='p50', description='MC stat:')
out_map = widgets.Output()

def render_map(*_):
    with out_map:
        clear_output()
        h = float(hour_view.value)
        if STATE.get('single_result') is not None:
            res = STATE['single_result']
            idx = int(round(h / float(res.meta['time_interval_hours'])))
            idx = max(0, min(idx, res.population_matrix.shape[0]-1))
            fig, ax = plt.subplots(1, 1, figsize=(10, 8))
            plot_map_snapshot(res.buildings, res.population_matrix[idx, :], ax=ax, title=f'Single snapshot @ {h:.1f}h')
            if STATE['boundary'] is not None:
                STATE['boundary'].plot(ax=ax, facecolor='none', edgecolor='cyan', linewidth=2)
            plt.show()
            return

        summ = STATE.get('mc_summary')
        if summ is None or summ.building_percentiles_by_hour is None:
            print('Run a simulation first.')
            return

        # Snapshots are stored only for key hours in MC config (default: 6,9,12,15,18,21)
        # Choose nearest stored hour.
        key_hours = np.array(list(summ.key_hours), dtype=float)
        nearest_i = int(np.argmin(np.abs(key_hours - h)))
        nearest_h = float(key_hours[nearest_i])

        stat_idx = {'p5': 0, 'p50': 1, 'p95': 2}[mc_stat.value]
        values = summ.building_percentiles_by_hour[nearest_i, stat_idx, :]

        fig, ax = plt.subplots(1, 1, figsize=(10, 8))
        plot_map_snapshot(STATE['buildings'], values, ax=ax, title=f'MC {mc_stat.value} snapshot @ {nearest_h:.1f}h')
        if STATE['boundary'] is not None:
            STATE['boundary'].plot(ax=ax, facecolor='none', edgecolor='cyan', linewidth=2)
        plt.show()

hour_view.observe(render_map, names='value')
mc_stat.observe(render_map, names='value')

display(widgets.HBox([hour_view, mc_stat]), out_map)
render_map()
